In [ ]:
import stackview as sv
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import os
import CtScanReader as ct

In [23]:
base_raw_path = '../../raw/'
base_alt_path = '../../altered/'
raw_scan_dirs = sorted(os.listdir(base_raw_path))
raw_scan_dirs.pop()
alt_scan_dirs = sorted(os.listdir(base_alt_path))

In [24]:
def write_manipulated_to_file(volume, scan, verbose = False):
    num = volume.shape[0]
    base_file = '../../unet/' + scan + '/'
    os.makedirs(base_file, exist_ok=True)
    if verbose == True:
        for i in tqdm(range(num)):
            file = '../../unet/' + scan + '/' + f'{scan}_{i+1:03d}.raw'
            volume[i,:,:].tofile(file)
    else:
        for i in range(num):
            file = '../../unet/' + scan + '/' + f'{scan}_{i+1:03d}.raw'
            volume[i,:,:].tofile(file)

In [25]:
import torch
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SliceDataset(Dataset):
    def __init__(self, X, Y):
        # Normalize to [0,1] if not already
        self.X = X.astype('float32') / X.max()
        self.Y = Y.astype('float32') / Y.max()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32).unsqueeze(0)  # (1,H,W)
        y = torch.tensor(self.Y[idx], dtype=torch.float32).unsqueeze(0)
        return x, y

print(device)

cuda


In [26]:
from torch import nn
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_filters=32):
        super().__init__()
        
        # Encoder
        self.enc1 = self.conv_block(in_channels, base_filters)
        self.enc2 = self.conv_block(base_filters, base_filters*2)
        self.enc3 = self.conv_block(base_filters*2, base_filters*4)
        self.enc4 = self.conv_block(base_filters*4, base_filters*8)
        
        # Bottleneck
        self.bottleneck = self.conv_block(base_filters*8, base_filters*16)
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(base_filters*16, base_filters*8, 2, stride=2)
        self.dec4 = self.conv_block(base_filters*16, base_filters*8)
        
        self.up3 = nn.ConvTranspose2d(base_filters*8, base_filters*4, 2, stride=2)
        self.dec3 = self.conv_block(base_filters*8, base_filters*4)
        
        self.up2 = nn.ConvTranspose2d(base_filters*4, base_filters*2, 2, stride=2)
        self.dec2 = self.conv_block(base_filters*4, base_filters*2)
        
        self.up1 = nn.ConvTranspose2d(base_filters*2, base_filters, 2, stride=2)
        self.dec1 = self.conv_block(base_filters*2, base_filters)
        
        self.out_conv = nn.Conv2d(base_filters, out_channels, 1)
    
    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        e4 = self.enc4(F.max_pool2d(e3, 2))
        
        b = self.bottleneck(F.max_pool2d(e4, 2))
        
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        return self.out_conv(d1)


In [27]:
from torch.utils.data import DataLoader, TensorDataset
def predict_streak_free_volume(model, volume, batch_size=2, device='cuda'):
    """
    Predicts a streak-free volume from a 3D CT volume using a trained U-Net.

    Args:
        model: trained U-Net model (residual U-Net preferred)
        volume: NumPy array of shape (N_slices, H, W)
        batch_size: batch size for inference
        device: 'cuda' or 'cpu'

    Returns:
        cleaned_volume: NumPy array of same shape as input (N, H, W)
    """
    model.to(device)
    model.eval()
    
    # Normalize and convert to tensor
    vol_tensor = torch.tensor(volume.astype('float32') / volume.max()).unsqueeze(1).to(device)  # (N,1,H,W)
    
    dataset = TensorDataset(vol_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    cleaned_slices = []
    with torch.no_grad():
        for (x_batch,) in loader:
            out = model(x_batch)  # residual U-Net: outputs corrected slice
            cleaned_slices.append(out.cpu().numpy())
    
    # Concatenate and remove channel dimension
    cleaned_volume = np.concatenate(cleaned_slices, axis=0)
    cleaned_volume = cleaned_volume.squeeze(1)  # shape: (N,H,W)
    
    # Optional: rescale to original intensity range
    cleaned_volume = (cleaned_volume * volume.max()).astype(volume.dtype)
    
    return cleaned_volume


In [28]:
model = UNet()
model.load_state_dict(torch.load("unet_epoch_10.pth"))

<All keys matched successfully>

In [29]:
scan = 'L008'
alt_scan = read_alt_scan(scan)
raw_scan = read_raw_scan(scan)

In [30]:
new_scan = predict_streak_free_volume(model, alt_scan)

In [31]:
combo = create_side_by_side(alt_scan, new_scan)

In [32]:
sv.slice(combo, slice_number=0, continuous_update=1)

In [33]:
alt_new_diff = new_scan - alt_scan
raw_new_diff = new_scan - raw_scan
diff_combo = create_side_by_side(alt_new_diff, raw_new_diff)

In [34]:
sv.slice(diff_combo, slice_number=0, continuous_update=1)

In [35]:
for i in tqdm(range(len(alt_scan_dirs))):
    scan = alt_scan_dirs[i]
    alt_scan = read_alt_scan(scan)
    new_scan = predict_streak_free_volume(model, alt_scan)
    write_manipulated_to_file(new_scan, scan)

100%|██████████| 20/20 [01:26<00:00,  4.35s/it]
